# Week 6 Practical — Sentiment Analysis with Naïve Bayes

**Dataset:** `week6_naive_bayes_sentiment_dataset.csv`  
**Labels:** `1 = positive`, `0 = negative`

### Learning objectives
Implement the full sentiment-analysis pipeline: preprocessing → frequencies → priors → conditional probabilities → Laplace smoothing → probability ratios → log-likelihood ratios → inference → validation → error analysis.

> **Important distinction:** Training learns the class prior and word λ values **once**. Validation/test samples reuse those learned parameters; they do not recompute them.

## 1. Setup and Dataset Exploration

In [ ]:
import re, math
import numpy as np
import pandas as pd
from collections import defaultdict

df = pd.read_csv("week6_naive_bayes_sentiment_dataset.csv")
rng = np.random.default_rng(42)
idx1=df.index[df.label==1].to_numpy(); idx0=df.index[df.label==0].to_numpy()
rng.shuffle(idx1); rng.shuffle(idx0)
c1=int(.8*len(idx1)); c0=int(.8*len(idx0))
train_idx=np.r_[idx1[:c1],idx0[:c0]]; val_idx=np.r_[idx1[c1:],idx0[c0:]]
rng.shuffle(train_idx); rng.shuffle(val_idx)
train_df=df.loc[train_idx].reset_index(drop=True)
val_df=df.loc[val_idx].reset_index(drop=True)
print("Train:",len(train_df),"Validation:",len(val_df))
train_df.head()

### TODO — Student Exercise

**Instructions**

Display the first 10 training examples. Count positive and negative examples and calculate their percentages. Print one example from each class. In a final code comment, explain whether the training set is approximately balanced and why class balance affects prior probabilities.

In [ ]:
# TODO: Write your solution here.

## 2. Bayes' Rule for Sentiment Classification

For class \(c\) and document \(d\),
$
P(c|d)=\frac{P(d|c)P(c)}{P(d)},\qquad P(c|d)\propto P(c)P(d|c).
$
For classification, \(P(d)\) is common to both classes, so it cancels when comparing them.

In [ ]:
p_pos, p_neg = 0.60, 0.40
p_d_given_pos, p_d_given_neg = 0.08, 0.02
# Posterior scores are proportional to likelihood × prior.

### TODO — Student Exercise

**Instructions**

Compute the unnormalized positive and negative posterior scores. Predict the class with the larger score. Add a comment explaining why P(document) can be omitted when comparing the two classes.

In [ ]:
# TODO: Write your solution here.

## 3. Text Preprocessing

In [ ]:
STOPWORDS={"a","an","the","this","that","is","are","was","were","be","being","been",
           "and","or","because","with","of","to","for","in","on","at","as","it",
           "i","am","my","your","our","their","very"}

def process_text(text, remove_stopwords=True):
    text=text.lower()
    text=re.sub(r"https?://\\S+|www\\.\\S+"," ",text)
    text=re.sub(r"@\\w+"," ",text)
    tokens=re.findall(r"[a-z]+(?:'[a-z]+)?",text)
    return [w for w in tokens if not remove_stopwords or w not in STOPWORDS]

print(process_text("This is not good, because your attitude is not even close to being nice."))

### TODO — Student Exercise

**Instructions**

Process three sentences of your choice and identify the sentiment-bearing tokens that remain. Then temporarily remove the word 'not' as if it were a stop word. Compare the outputs and explain why removing negation can reverse the intended sentiment.

In [ ]:
# TODO: Write your solution here.

## 4. Class-Specific Word Frequencies

In [ ]:
def build_freqs(texts,labels):
    freqs=defaultdict(lambda:{0:0,1:0})
    for text,label in zip(texts,labels):
        for word in process_text(text):
            freqs[word][int(label)]+=1
    return freqs

freqs=build_freqs(train_df.text,train_df.label)
for w in ["happy","good","bad","terrible","not"]:
    print(w,dict(freqs[w]))

### TODO — Student Exercise

**Instructions**

Choose at least five words: two expected positive words, two expected negative words, and one neutral/common word. Create a DataFrame with columns word, Pos, and Neg. Explain whether the observed frequencies match your expectations.

In [ ]:
# TODO: Write your solution here.

## 5. Class Priors and Log-Prior

$
P(pos)=\frac{N_{pos}}{N_{docs}},\quad P(neg)=\frac{N_{neg}}{N_{docs}},\quad
\text{logprior}=\log\frac{P(pos)}{P(neg)}.
$

In [ ]:
n_pos=(train_df.label==1).sum(); n_neg=(train_df.label==0).sum()
p_pos=n_pos/len(train_df); p_neg=n_neg/len(train_df)
logprior=math.log(p_pos/p_neg)
print(p_pos,p_neg,logprior)

### TODO — Student Exercise

**Instructions**

Recompute P(pos), P(neg), and logprior from the training data. Verify that the priors sum to 1. Explain what zero, positive, and negative logprior values mean before any words in a new document are examined.

In [ ]:
# TODO: Write your solution here.

## 6. Conditional Probabilities and Zero Frequencies

$
P(w|c)=\frac{\mathrm{freq}(w,c)}{N_c}.
$
A single zero word probability can make the entire product likelihood zero.

In [ ]:
N_pos=sum(v[1] for v in freqs.values())
N_neg=sum(v[0] for v in freqs.values())
vocab=sorted(freqs)
V=len(vocab)
print("N_pos =",N_pos,"N_neg =",N_neg,"V =",V)

### TODO — Student Exercise

**Instructions**

Find a vocabulary word that occurs in one class but zero times in the other. Compute its unsmoothed P(word|pos) and P(word|neg). Explain what a zero conditional probability does to a product of document word probabilities.

In [ ]:
# TODO: Write your solution here.

## 7. Laplace Smoothing

$
P(w|c)=\frac{\mathrm{freq}(w,c)+1}{N_c+V}.
$
Here \($N_c$\) is the number of token occurrences in class \(c\), and \(V\) is vocabulary size.

In [ ]:
def p_smooth(word,label):
    N=N_pos if label==1 else N_neg
    return (freqs[word][label]+1)/(N+V)

for w in ["happy","good","bad","terrible"]:
    print(w,round(p_smooth(w,1),4),round(p_smooth(w,0),4))

### TODO — Student Exercise

**Instructions**

Use the zero-frequency word from the previous section and compute its smoothed probabilities in both classes. Verify neither is zero. Sum P(word|class) across the entire vocabulary for one class and verify the result is approximately 1. Explain why the denominator adds V.

In [ ]:
# TODO: Write your solution here.

## 8. Probability Ratios

$
\mathrm{ratio}(w)=\frac{P(w|pos)}{P(w|neg)}.
$
Values above 1 favor positive; below 1 favor negative; values near 1 are approximately neutral.

In [ ]:
def ratio(word):
    return p_smooth(word,1)/p_smooth(word,0)

for w in ["great","happy","good","bad","terrible","not"]:
    if w in freqs: print(w,round(ratio(w),3))

### TODO — Student Exercise

**Instructions**

Compute the smoothed probability ratio for every vocabulary word. Display the 10 largest and 10 smallest ratios. Identify which favor positive versus negative sentiment and discuss one surprising result, if any.

In [ ]:
# TODO: Write your solution here.

## 9. Log-Likelihood Ratios (Lambda)

$
\lambda(w)=\log\frac{P(w|pos)}{P(w|neg)}.
$
Positive λ favors positive sentiment; negative λ favors negative sentiment.

In [ ]:
lambdas={}
for w in vocab:
    lambdas[w]=math.log(p_smooth(w,1)/p_smooth(w,0))

for w in ["great","happy","good","bad","terrible","not"]:
    if w in lambdas: print(w,round(lambdas[w],3))

### TODO — Student Exercise

**Instructions**

Create a DataFrame containing word, P(w|pos), P(w|neg), ratio, and lambda. Sort by lambda and display the five strongest positive and five strongest negative words. Numerically verify lambda = log(ratio), then explain why log-space is useful.

In [ ]:
# TODO: Write your solution here.

## 10. Naive Bayes Inference

$
\mathrm{score}(d)=\log\frac{P(pos)}{P(neg)}+\sum_{w\in d}\lambda(w).
$
Predict positive if score \(>0\), otherwise negative.

In [ ]:
def predict_score(text):
    score=logprior
    for w in process_text(text):
        if w in lambdas: score+=lambdas[w]
    return score

def predict(text):
    score=predict_score(text)
    return score,int(score>0)

for text in ["I loved this excellent movie","This service was awful and disappointing"]:
    print(text,predict(text))

### TODO — Student Exercise

**Instructions**

Trace the prediction for 'The movie was great and wonderful.' Print the processed tokens and lambda contribution of every known token. Add them to logprior, print the final score and class, and verify your result with predict_score(). Identify the strongest contributing word.

In [ ]:
# TODO: Write your solution here.

## 11. Validation and Accuracy

$
\mathrm{Accuracy}=\frac{1}{m}\sum_{i=1}^{m}\mathbf{1}(\hat y^{(i)}=y^{(i)}).
$

In [ ]:
scores=np.array([predict_score(t) for t in val_df.text])
pred=(scores>0).astype(int)
y_val=val_df.label.to_numpy()
accuracy=np.mean(pred==y_val)
print("Validation accuracy:",round(float(accuracy),4))

### TODO — Student Exercise

**Instructions**

Recompute all validation predictions yourself and calculate accuracy without a library metric. Compute TP, TN, FP, and FN. Print at least three misclassified examples if available, inspect their tokens, and propose a plausible reason for each error. Remember: logprior and lambda are learned once from training data, not recomputed per validation sample.

In [ ]:
# TODO: Write your solution here.

## 12. Processing Error: Punctuation

In [ ]:
examples=["Great.","Great!!!","Great :)"]
for x in examples: print(x,"->",process_text(x))

### TODO — Student Exercise

**Instructions**

Compare the processed representations of the three examples. Explain which intensity or sentiment cues disappear. Propose and implement one extra feature that preserves useful punctuation or emoticon information, such as exclamation count.

In [ ]:
# TODO: Write your solution here.

## 13. Processing Error: Removing Words / Negation

In [ ]:
examples=["This is good.","This is not good."]
for x in examples: print(x,process_text(x),predict(x))

### TODO — Student Exercise

**Instructions**

Create an alternative preprocessing function that incorrectly removes 'not'. Compare tokens and model scores for the two sentences. Explain why negation should often be retained. As an extension, create a NOT_good feature and explain how it preserves local meaning.

In [ ]:
# TODO: Write your solution here.

## 14. Processing Error: Word Order

In [ ]:
examples=["I am happy because I did not go.",
          "I am not happy because I did go."]
for x in examples: print(x,process_text(x),predict(x))

### TODO — Student Exercise

**Instructions**

Compare the token sets, scores, and predictions. Explain why unigram Naive Bayes may struggle even though the sentences have different meanings. Construct bigram-style features such as not_happy or not_go and explain how they retain more local word order.

In [ ]:
# TODO: Write your solution here.

## 15. Sarcasm, Irony, and Adversarial Language

In [ ]:
sarcastic="Great, another app crash. Exactly what I needed!"
print(process_text(sarcastic))
print(predict(sarcastic))

### TODO — Student Exercise

**Instructions**

Print each known token's lambda contribution, final score, and prediction for the sarcastic sentence. Decide whether the prediction matches the intended sentiment. Rewrite the sentence using literal negative language, classify it again, and explain in 3–5 sentences why sarcasm is difficult for bag-of-words Naive Bayes.

In [ ]:
# TODO: Write your solution here.

## 16. Naive Bayes Assumptions

In [ ]:
# Conditional independence is a simplifying assumption:
# P(w1,...,wm | class) ≈ product_i P(wi | class)

### TODO — Student Exercise

**Instructions**

Give an example of two words whose sentiment depends strongly on their combination. Explain why treating them independently can be misleading. Then choose a new deployment domain and identify two corpus/distribution changes that could hurt this model. Suggest one mitigation.

In [ ]:
# TODO: Write your solution here.

## 17. Final Challenge: Build Naive Bayes End-to-End

In [ ]:
def train_naive_bayes(texts,labels):
    # TODO: learn frequencies, vocabulary, token totals,
    # priors/logprior, smoothed probabilities, and lambdas.
    pass

def predict_naive_bayes(text,logprior,lambdas):
    # TODO: preprocess, accumulate log score, and return score + label.
    pass

### TODO — Student Exercise

**Instructions**

Complete both functions without using a prebuilt Naive Bayes classifier. Train only on train_df and evaluate on val_df. Print validation accuracy and five rows containing text, true label, score, and predicted label. Finish with a short reflection: Why smoothing? Why logs? Which preprocessing choice is risky? What is one major Naive Bayes limitation?

In [ ]:
# TODO: Write your solution here.

# Practical Summary
You implemented and investigated:

**text → preprocessing → word frequencies → class priors → conditional probabilities → Laplace smoothing → ratios → λ → log-prior + λ summation → prediction → validation**

You also examined why **negation, punctuation, word order, corpus shift, sarcasm, irony, and euphemisms** can cause errors.

### Key rule
`logprior` and the word-level `lambdas` are **model parameters learned from the training set**. For each validation/test sample, only the document score is newly computed.